# Day 003 — Doubly Linked List · Insertion Sort · Binary Search
**Date:** 2026-08-04  |  **Difficulty:** Beginner-Intermediate  |  **Series:** Daily DSA

---

## What You Will Learn Today
| Topic | Concept | Time Complexity |
|-------|---------|----------------|
| Data Structure | **Doubly Linked List** | Insert/delete at head or tail O(1) |
| Sorting | **Insertion Sort** | Best O(n), Worst O(n²) — stable & adaptive |
| Searching | **Binary Search** | O(log n) — requires sorted input |

> **Series recap**
> - Day 001: Array (O(1) access) · Bubble Sort · Linear Search O(n)
> - Day 002: Singly Linked List (O(1) head ops) · Selection Sort · Jump Search O(√n)
> - **Today**: Doubly Linked List (O(1) head AND tail ops) · Insertion Sort · Binary Search **O(log n)**

> Run each cell top-to-bottom with **Shift+Enter** to follow along interactively.

---
## PART 1 — Data Structure: Doubly Linked List

A **doubly linked list** extends the singly linked list by giving each node a `prev` pointer in addition to `next`.  
This allows **O(1) traversal in both directions** and O(1) deletion without needing the previous node.

### Memory layout
```
         head                                              tail
          │                                                 │
          ▼                                                 ▼
None ◀─┬──────┬──┐    ┌──┬──────┬──┐    ┌──┬──────┬──┐    ┌──┬──────┬──▶ None
       │  10  │●─┼───▶│●─│  20  │●─┼───▶│●─│  30  │●─┼───▶│●─│  40  │
       └──────┴──┘    └──┴──────┴──┘    └──┴──────┴──┘    └──┴──────┘
         prev  next     prev  next        prev  next        prev  next
```

### Singly vs Doubly Linked List
| Operation | Singly LL | Doubly LL |
|-----------|-----------|----------|
| Insert at head | O(1) | O(1) |
| Insert at tail | O(1)* | O(1) |
| Delete at head | O(1) | O(1) |
| Delete at tail | O(n)** | **O(1)** |
| Delete a given node | O(n) (need prev) | **O(1)** (has prev pointer) |
| Reverse traversal | O(n) (must restart) | **O(1)** (follow prev) |
| Memory per node | 1 pointer | 2 pointers |

\* with tail pointer  
\** even with tail pointer — need to update tail to its predecessor

In [ ]:
# ─── Doubly Linked List ───────────────────────────────────────────────────────

class DNode:
    """A doubly-linked node holding data, a prev pointer, and a next pointer."""
    __slots__ = ('data', 'prev', 'next')

    def __init__(self, data):
        self.data = data
        self.prev = None
        self.next = None

    def __repr__(self):
        return f'DNode({self.data})'


class DoublyLinkedList:
    """
    Doubly Linked List with head and tail sentinels.

    Using sentinel (dummy) nodes at both ends simplifies every edge case:
    - The list is never truly empty — head.next is always valid.
    - Insert/delete code never needs to special-case None pointers.
    """

    def __init__(self):
        # Sentinel nodes — they hold no real data
        self._head = DNode(None)   # dummy head
        self._tail = DNode(None)   # dummy tail
        self._head.next = self._tail
        self._tail.prev = self._head
        self._size = 0

    # ── Internal helper ───────────────────────────────────────────────────────
    def _insert_between(self, data, before, after):
        """Insert a new node with `data` between existing nodes `before` and `after`."""
        node = DNode(data)
        node.prev, node.next = before, after
        before.next = after.prev = node
        self._size += 1
        return node

    def _delete_node(self, node):
        """Unlink `node` from the list — O(1) because we have its prev pointer."""
        node.prev.next = node.next
        node.next.prev = node.prev
        node.prev = node.next = None   # help GC
        self._size -= 1
        return node.data

    # ── O(1) public API ───────────────────────────────────────────────────────
    def prepend(self, data):
        """Insert at the logical head — O(1)."""
        return self._insert_between(data, self._head, self._head.next)

    def append(self, data):
        """Insert at the logical tail — O(1)."""
        return self._insert_between(data, self._tail.prev, self._tail)

    def pop_head(self):
        """Remove and return the first real node's data — O(1)."""
        if self._size == 0:
            raise IndexError('pop from empty list')
        return self._delete_node(self._head.next)

    def pop_tail(self):
        """Remove and return the last real node's data — O(1). (Impossible in SLL!)"""
        if self._size == 0:
            raise IndexError('pop from empty list')
        return self._delete_node(self._tail.prev)

    # ── O(n) public API ───────────────────────────────────────────────────────
    def find(self, target):
        """Return the first DNode whose data == target, or None — O(n)."""
        current = self._head.next
        while current is not self._tail:
            if current.data == target:
                return current
            current = current.next
        return None

    def delete_value(self, target):
        """Delete first node with data == target — O(n) to find, O(1) to remove."""
        node = self.find(target)
        if node is None:
            raise ValueError(f'{target!r} not found')
        return self._delete_node(node)

    def insert_after_value(self, target, new_data):
        """Insert new_data right after the first node with data == target — O(n)."""
        node = self.find(target)
        if node is None:
            raise ValueError(f'{target!r} not found')
        return self._insert_between(new_data, node, node.next)

    def to_list(self):
        """Return elements as a plain Python list (head → tail) — O(n)."""
        result, current = [], self._head.next
        while current is not self._tail:
            result.append(current.data)
            current = current.next
        return result

    def to_list_reversed(self):
        """Return elements as a plain Python list (tail → head) — O(n)."""
        result, current = [], self._tail.prev
        while current is not self._head:
            result.append(current.data)
            current = current.prev
        return result

    def __len__(self):  return self._size
    def __repr__(self): return 'None ⟵ ' + ' ⟷ '.join(str(x) for x in self.to_list()) + ' ⟶ None'

In [ ]:
# ─── Demo ─────────────────────────────────────────────────────────────────────
dll = DoublyLinkedList()

print("--- Building ---")
for v in [20, 30, 40]:
    dll.append(v)
dll.prepend(10)
print(f"append(20,30,40) + prepend(10) : {dll}")
print(f"  size={len(dll)}")

print("\n--- O(1) tail operations (impossible in SLL without O(n)) ---")
tail_val = dll.pop_tail()
print(f"pop_tail() → {tail_val}   list now: {dll}")
dll.append(tail_val)   # put it back

print("\n--- Bidirectional traversal ---")
print(f"Forward  : {dll.to_list()}")
print(f"Backward : {dll.to_list_reversed()}")

print("\n--- Insert and delete ---")
dll.insert_after_value(20, 25)
print(f"insert_after_value(20, 25) : {dll}")
dll.delete_value(25)
print(f"delete_value(25)           : {dll}")

print("\n--- pop_head ---")
print(f"pop_head() → {dll.pop_head()}   list now: {dll}")

---
## PART 2 — Sorting Algorithm: Insertion Sort

### Core Idea
Think of sorting a hand of playing cards.  
You pick up one card at a time and **insert it into its correct position** among the already-sorted cards in your hand.

```
Hand (sorted so far)   New card   Action
─────────────────────────────────────────
[ ]                      5        just place it
[5]                      2        shift 5 right → insert 2 at 0
[2, 5]                   4        shift 5 right → insert 4 at 1
[2, 4, 5]                1        shift 5,4,2 right → insert 1 at 0
[1, 2, 4, 5]             3        shift 5,4 right → insert 3 at 2
[1, 2, 3, 4, 5]          ✓ done
```

### Step-by-step on `[5, 2, 4, 1, 3]`
```
i=1: key=2, shift 5 right  → [2, 5, 4, 1, 3]
i=2: key=4, shift 5 right  → [2, 4, 5, 1, 3]
i=3: key=1, shift 5,4,2    → [1, 2, 4, 5, 3]
i=4: key=3, shift 5,4      → [1, 2, 3, 4, 5] ✓
```

### Why Insertion Sort matters
- **Adaptive:** O(n) on nearly-sorted data — the inner loop terminates early when the key is in place.
- **Stable:** equal elements keep their original order.
- **Online:** can sort a stream of arriving elements without seeing them all first.
- **Used in practice:** Python's `timsort` and C++'s `introsort` fall back to Insertion Sort for small sub-arrays.

### Complexity
| Case | Time | Space | Shifts |
|------|------|-------|--------|
| Best (already sorted) | O(n) | O(1) | 0 |
| Average | O(n²) | O(1) | O(n²/4) |
| Worst (reverse sorted) | O(n²) | O(1) | O(n²/2) |

In [ ]:
# ─── Insertion Sort ───────────────────────────────────────────────────────────

def insertion_sort(arr, verbose=False):
    """
    Sort `arr` in-place using Insertion Sort.

    Algorithm:
      for i in 1..n-1:
          key = arr[i]
          j = i - 1
          while j >= 0 and arr[j] > key:
              arr[j+1] = arr[j]   # shift right
              j -= 1
          arr[j+1] = key          # insert key in its correct slot

    Args:
        arr     : list of comparable elements
        verbose : print each insertion step if True

    Returns:
        (sorted list, shift_count, comparison_count)
    """
    a = arr[:]          # work on a copy
    n = len(a)
    shifts = comparisons = 0

    for i in range(1, n):
        key = a[i]
        j = i - 1

        # Shift elements of a[0..i-1] that are greater than `key` one position right
        while j >= 0 and a[j] > key:
            comparisons += 1
            a[j + 1] = a[j]
            j -= 1
            shifts += 1

        if j >= 0:
            comparisons += 1    # the final comparison that stopped the loop

        a[j + 1] = key          # place key in its correct slot

        if verbose:
            print(f"  i={i}: key={key:>3}, j ends at {j:>2}, shifts={shifts:>3}  → {a}")

    return a, shifts, comparisons

In [ ]:
# ─── Verbose walkthrough ──────────────────────────────────────────────────────
sample = [5, 2, 4, 1, 3]
print(f"Input : {sample}")
sorted_arr, shifts, cmps = insertion_sort(sample, verbose=True)
print(f"Output: {sorted_arr}")
print(f"Stats : {shifts} shifts, {cmps} comparisons")

print()
already = [1, 2, 3, 4, 5]
print(f"Already sorted {already}:")
_, shifts2, cmps2 = insertion_sort(already, verbose=True)
print(f"Stats : {shifts2} shifts (best case — adaptive!)")

In [ ]:
# ─── Test Cases ───────────────────────────────────────────────────────────────
test_cases = [
    ([5, 2, 4, 1, 3],        "random"),
    ([1, 2, 3, 4, 5],        "already sorted (best case — O(n))"),
    ([5, 4, 3, 2, 1],        "reverse sorted (worst case — O(n²))"),
    ([42],                   "single element"),
    ([],                     "empty list"),
    ([3, 3, 1, 1, 2],        "with duplicates (stable)"),
    ([-4, 0, 7, -1, 3],      "with negatives"),
    ([9, 1, 2, 3, 4, 5, 6],  "nearly sorted — one out-of-place"),
]

print(f"{'Input':<32} {'Sorted':<32} {'Shifts':>6}  {'Case'}")
print("-" * 90)
for data, label in test_cases:
    result, shifts, _ = insertion_sort(data)
    print(f"{str(data):<32} {str(result):<32} {shifts:>6}  {label}")

---
## PART 3 — Searching Algorithm: Binary Search

### Core Idea
On a **sorted** array, compare the target to the **middle element**.  
- If equal → found.  
- If target < mid → search the **left half**.  
- If target > mid → search the **right half**.  

Each comparison **halves** the search space, giving O(log n).

### Step-by-step on `[1, 3, 5, 7, 9, 11, 13]`, target = 7
```
lo=0  hi=6  mid=3  arr[3]=7  → FOUND at index 3 ✓
```

### Step-by-step on the same array, target = 11
```
lo=0  hi=6  mid=3  arr[3]=7  < 11  → search right half
lo=4  hi=6  mid=5  arr[5]=11 = 11  → FOUND at index 5 ✓
```

### Step-by-step on the same array, target = 6 (not present)
```
lo=0  hi=6  mid=3  arr[3]=7  > 6   → search left half
lo=0  hi=2  mid=1  arr[1]=3  < 6   → search right half
lo=2  hi=2  mid=2  arr[2]=5  < 6   → search right half
lo=3  hi=2  lo > hi → NOT FOUND → return -1
```

### Complexity
| Case | Time | Space (iterative) | Space (recursive) |
|------|------|-------------------|-------------------|
| Best (target at mid) | O(1) | O(1) | O(1) |
| Average | O(log n) | O(1) | O(log n) |
| Worst | O(log n) | O(1) | O(log n) |

> **Comparison**: Linear O(n), Jump O(√n), Binary **O(log n)** — Binary wins on large sorted data.

In [ ]:
# ─── Binary Search — iterative (preferred: O(1) space) ────────────────────────

def binary_search(arr, target, verbose=False):
    """
    Search for `target` in a SORTED array using iterative Binary Search.

    Args:
        arr     : sorted list of comparable elements
        target  : value to find
        verbose : print each step if True

    Returns:
        int : index of target, or -1 if not found
    """
    lo, hi = 0, len(arr) - 1
    step = 0

    while lo <= hi:
        step += 1
        # Use lo + (hi - lo) // 2 instead of (lo + hi) // 2
        # to avoid integer overflow in languages with fixed-width integers.
        mid = lo + (hi - lo) // 2

        if verbose:
            print(f"  Step {step}: lo={lo}, hi={hi}, mid={mid}, arr[mid]={arr[mid]}")

        if arr[mid] == target:
            if verbose: print(f"  → FOUND at index {mid}")
            return mid
        elif arr[mid] < target:
            if verbose: print(f"  → {arr[mid]} < {target}, search right half")
            lo = mid + 1
        else:
            if verbose: print(f"  → {arr[mid]} > {target}, search left half")
            hi = mid - 1

    if verbose: print(f"  → lo={lo} > hi={hi}: NOT FOUND")
    return -1


# ─── Binary Search — recursive (elegant, but uses O(log n) stack space) ───────

def binary_search_recursive(arr, target, lo=0, hi=None):
    """
    Recursive variant — same logic, illustrates the divide-and-conquer structure.
    """
    if hi is None:
        hi = len(arr) - 1
    if lo > hi:
        return -1
    mid = lo + (hi - lo) // 2
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, hi)
    else:
        return binary_search_recursive(arr, target, lo, mid - 1)

In [ ]:
# ─── Verbose walkthrough ──────────────────────────────────────────────────────
arr = [1, 3, 5, 7, 9, 11, 13]
print(f"Array: {arr}\n")

print("Search for 7 (present):")
binary_search(arr, 7, verbose=True)

print("\nSearch for 11 (present):")
binary_search(arr, 11, verbose=True)

print("\nSearch for 6 (absent):")
binary_search(arr, 6, verbose=True)

print("\n--- Recursive variant ---")
print(f"binary_search_recursive(arr, 11) → {binary_search_recursive(arr, 11)}")
print(f"binary_search_recursive(arr, 6)  → {binary_search_recursive(arr, 6)}")

In [ ]:
# ─── Test Cases ───────────────────────────────────────────────────────────────
search_tests = [
    ([1,3,5,7,9,11,13], 7,   "found in middle"),
    ([1,3,5,7,9,11,13], 1,   "found at start"),
    ([1,3,5,7,9,11,13], 13,  "found at end"),
    ([1,3,5,7,9,11,13], 6,   "not found (between elements)"),
    ([1,3,5,7,9,11,13], 99,  "not found (beyond end)"),
    ([42],               42,  "single element — found"),
    ([42],               1,   "single element — not found"),
    ([],                 5,   "empty list"),
    (list(range(1, 1001)), 756, "1000-element range"),
    (list(range(1, 1001)), 1001, "1000-element range — not found"),
]

print(f"{'Array (preview)':<30} {'Target':>7}  {'Result':<14} {'Case'}")
print("-" * 80)
for arr, target, label in search_tests:
    result = binary_search(arr, target)
    preview = str(arr[:4]) + ('...' if len(arr) > 4 else '')
    found   = f"index {result}" if result != -1 else "not found"
    print(f"{preview:<30} {str(target):>7}  {found:<14} {label}")

---
## PART 4 — Putting It All Together

Real-world mini-pipeline:
1. Maintain a **Doubly Linked List** of sensor readings (fast append + pop from either end)
2. Sort readings with **Insertion Sort** (ideal because sensor streams are nearly sorted)
3. Find a threshold reading with **Binary Search**

In [ ]:
# ─── End-to-End Example ───────────────────────────────────────────────────────
import random
random.seed(3)

# 1. Sensor readings arrive in a mostly-sorted stream (small random noise)
base = list(range(10, 110, 10))                            # [10,20,...,100]
readings_raw = [v + random.randint(-3, 3) for v in base]  # add small jitter

sensor_log = DoublyLinkedList()
for r in readings_raw:
    sensor_log.append(r)

print("1. Sensor log (DLL, arrival order):")
print("  ", sensor_log)
print(f"   head={sensor_log._head.next.data}  "
      f"tail={sensor_log._tail.prev.data}  size={len(sensor_log)}")

# 2. Sort with Insertion Sort — adaptive, great for nearly-sorted data
sorted_readings, shifts, cmps = insertion_sort(sensor_log.to_list())
print(f"\n2. After Insertion Sort ({shifts} shifts — low because nearly sorted!):")
print("  ", sorted_readings)

# 3. Binary Search for alarm threshold
ALARM = 70
pos = binary_search(sorted_readings, ALARM)
if pos != -1:
    print(f"\n3. Binary Search: alarm threshold {ALARM} found at sorted index {pos}")
else:
    first_over = next((v for v in sorted_readings if v >= ALARM), None)
    print(f"\n3. Exact value {ALARM} not present. First reading ≥ {ALARM}: {first_over}")
    pos2 = binary_search(sorted_readings, first_over)
    print(f"   Found at index {pos2}")

# 4. DLL O(1) tail pop — discard oldest reading after processing
oldest = sensor_log.pop_head()
print(f"\n4. Popped oldest reading from DLL head: {oldest}")
print(f"   DLL size now: {len(sensor_log)}")

---
## Complexity Cheat Sheet

```
┌──────────────────────────────┬──────────┬──────────┬──────────┬─────────┐
│ Operation                    │ Best     │ Average  │ Worst    │ Space   │
├──────────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ DLL prepend / append         │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
│ DLL pop_head / pop_tail      │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
│ DLL delete (given node)      │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
│ DLL find / delete by value   │ O(n)     │ O(n)     │ O(n)     │ O(1)   │
├──────────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Insertion Sort               │ O(n)     │ O(n²)    │ O(n²)    │ O(1)   │
│   (shifts)                   │ 0        │ O(n²/4)  │ O(n²/2)  │        │
├──────────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Binary Search (iterative)    │ O(1)     │ O(log n) │ O(log n) │ O(1)   │
│ Binary Search (recursive)    │ O(1)     │ O(log n) │ O(log n) │ O(log n)│
└──────────────────────────────┴──────────┴──────────┴──────────┴─────────┘
```

## Key Takeaways
- Doubly Linked List pays one extra pointer per node to unlock **O(1) tail deletion** and **O(1) any-node deletion** — SLL can't match either.
- Insertion Sort's **adaptive** nature (O(n) best case) makes it Python's/C++'s choice for small or nearly-sorted sub-arrays inside Timsort/Introsort.
- Binary Search's **O(log n)** is the gold standard for sorted lookup — halving the search space means 1 billion elements need at most 30 comparisons.

---
**Tomorrow — Day 004:** Stack · Shell Sort · Sentinel Search